## Import Libraries

In [0]:
from pyspark.sql.functions import col

## Schema and Variables

In [0]:
_schema = """ 
            txn_time TIMESTAMP,
            order_number STRING, 
            cust_id INT, 
            order_value DOUBLE
        """

In [0]:
schema_location = "/Volumes/workspace/learning/learning_volume/AutoLoader/customer/schema"
checkpoint_location = "/Volumes/workspace/learning/learning_volume/AutoLoader/customer/checkpoint"
landing_location = "/Volumes/workspace/learning/learning_volume/AutoLoader/customer/landing/"
target_schema = "curated"
target_table = "autoloader_cust_orders"

## Autoloader Job for Incremental data processing

In [0]:
df = (spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header","true")
            .option("cloudFiles.schemaLocation", schema_location)
            .schema(_schema)
            .option("cloudFiles.schemaEvolutionMode", "rescue")
            .load(landing_location)
    )

df_final = df.withColumn("file_name" , col("_metadata.file_path"))

(
    df_final.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_location)
    .trigger(availableNow=True)
    .toTable(f"{target_schema}.{target_table}")
)

## Verify count after 2 runs

In [0]:
%sql
SELECT COUNT(1) FROM curated.autoloader_cust_orders;

COUNT(1)
12


## Test
- Upload file and manual run
- Verify records
- Upload another file and run
- Only rows in new file will be appended 

## Assumption: due to some reason checkpoint got corrupted
- Delete checkpoint directory
- Recreate checkpoint directory
- Rerun the job

In [0]:
dbutils.fs.rm(checkpoint_location,True)

True

## Entire table got duplicated
- All files from landing again processed and loaded to target table

In [0]:
%sql
SELECT COUNT(1) FROM curated.autoloader_cust_orders;

COUNT(1)
12


## Lets clean the table
- Restore to last version before checkpoint got corrupt

In [0]:
%sql
DESCRIBE HISTORY curated.autoloader_cust_orders;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-02-22T06:01:50.000Z,3587565026941240,sayon.biems@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> e3631e11-38a2-4ba6-a2ce-c2886f0c971d, epochId -> 0, statsOnLoad -> true)",null,List(693507180640845),505373b2-cc28-4e5a-9e02-fa05de2933c1,0222-053522-2f89ndx3-v2n,2,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 6, numOutputBytes -> 2203, numAddedFiles -> 1)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
2,2026-02-22T06:00:22.000Z,3587565026941240,sayon.biems@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 527bb79c-f5df-4295-ab28-24c9a90ede9f, epochId -> 1, statsOnLoad -> true)",null,List(693507180640845),91b2ad87-cc24-441d-b069-b7d58a282e1d,0222-053522-2f89ndx3-v2n,1,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 3, numOutputBytes -> 2106, numAddedFiles -> 1)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
1,2026-02-22T05:57:24.000Z,3587565026941240,sayon.biems@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 527bb79c-f5df-4295-ab28-24c9a90ede9f, epochId -> 0, statsOnLoad -> true)",null,List(693507180640845),e5270939-6d1c-4dce-ae9f-88ab1bd8971b,0222-053522-2f89ndx3-v2n,0,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 3, numOutputBytes -> 2107, numAddedFiles -> 1)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
0,2026-02-22T05:57:18.000Z,3587565026941240,sayon.biems@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.enableDeletionVectors"":""true"",""delta.writePartitionColumnsToParquet"":""true"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-60d1c002-f77d-420a-8f11-f537a58562d3"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-8cc220a4-3bfc-4867-8af5-f05d932ed31b""}, statsOnLoad -> false)",null,List(693507180640845),e5270939-6d1c-4dce-ae9f-88ab1bd8971b,0222-053522-2f89ndx3-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13


## Restore to version 2

In [0]:
%sql
RESTORE TABLE curated.autoloader_cust_orders TO VERSION AS OF 2;

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
4213,2,1,0,2203,0


## State of table restored

In [0]:
%sql
SELECT COUNT(1) FROM curated.autoloader_cust_orders;

COUNT(1)
6


## Add guradrails and create Idempotent pipeline

In [0]:
df = (spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header","true")
            .option("cloudFiles.schemaLocation", schema_location)
            .schema(_schema)
            .option("cloudFiles.schemaEvolutionMode", "rescue")
            .load(landing_location)
    )

df_final = df.withColumn("file_name" , col("_metadata.file_path"))

def merge_into_target(input_df, batch_id):

    input_df.createOrReplaceTempView("delta_data")

    merge_query = f"""
    MERGE INTO {target_schema}.{target_table} tgt
    USING delta_data src
    ON tgt.file_name = src.file_name
    WHEN NOT MATCHED THEN INSERT *
    """
    
    spark.sql(merge_query)

(
    df_final.writeStream
    .format("delta")
    .foreachBatch(merge_into_target)
    .outputMode("append")
    .option("checkpointLocation", checkpoint_location)
    .trigger(availableNow=True)
    .start()
)

## Run the above block without uploading any new file
- It should not process any file from landing

## Delete Checkpoint again and test
- It should not process any file from landing

In [0]:
%sql
SELECT COUNT(*) FROM curated.autoloader_cust_orders;

COUNT(*)
6


## Upload a new file and test
- My new file has 2 rows 
- Only 2 added to table. 8 rows now

In [0]:
%sql
SELECT COUNT(*) FROM curated.autoloader_cust_orders;

COUNT(*)
8


## Conclusion
- Using the landing file name is a good design option.
- If same filename is picked from landing after checkpoint is deleted then I will not process it.
- Process only new files even if checkpoint is deleted.
- After data got duplicated in table STOP the streaming job and work on damage control.
- Restore to latest correct version
- Add guardrails and restrart the Job
- Checkpoint can fail due to many reasons - Logic change, Join condition change, tables added or removed in Joins
- Using Merge and Auto Loader we can desing a checkpoint failure and Idempotent Pipeline